# Avaliação do D3 (Alfred + servidor MCP)

Avalia o Alfred de ponta a ponta, através do servidor MCP, com as métricas dos slides "LLM-based Agents" e o guia do Ragas [*evaluate and improve RAG*](https://docs.ragas.io/en/stable/howtos/applications/evaluate-and-improve-rag/).

O MCP é avaliado a dois níveis:

1. **Cada ferramenta sozinha** (secção 3, sem LLM): chama-se cada ferramenta diretamente e confere-se o resultado, a fiabilidade e o tempo.
2. **O agente a usar as ferramentas** (secções 4 e 5): o Alfred responde às perguntas do ground truth e avaliam-se as chamadas e as respostas.

| Grupo | Métricas | Pergunta a que responde |
|---|---|---|
| Ferramentas | ToolHit, ToolRecall, ToolPrecision, NumToolCalls | o agente escolheu as ferramentas certas? |
| Servidor MCP | TesteCorreto, TesteRespondeu (testes diretos); TaxaSucesso, LatenciaMedia (nas respostas) | as ferramentas funcionam, são fiáveis e rápidas? |
| Correção | Correctness (pass/fail), JudgeScore (0-10), AnswerRelevancy, BLEU, ROUGE-L | a resposta está certa? |
| Fundamentação | Faithfulness | a resposta baseia-se no que as ferramentas devolveram? |
| Falhas | cada "fail" recebe uma causa: seleção de ferramenta, argumentos, ferramenta falhou, limite de chamadas ou geração | porque é que falhou? |

A recuperação (P@k, R@k, Hit Rate@k, MRR) já foi avaliada no D2; aqui avalia-se o agente.

Perguntas: as 15 do D2 (`rag/evaluation/ground_truth_example.json`, categoria `arquivos`) mais as do D3 (`ground_truth_d3.json`: web, sistemas da Batcaverna e Batman).

Cada pergunta corre nos dois modos do agente, e a secção 7 compara-os:

- **`rag_primeiro`**: o código consulta sempre o RAG (`relevancia` + `pesquisar_arquivos`) antes do LLM e põe os excertos no prompt; o LLM só acrescenta ferramentas se precisar.
- **`livre`**: RAG agêntico; o LLM decide todas as ferramentas, arquivos incluídos.

Notas:

- Rápido (sem LLM): testes diretos às ferramentas, ferramentas, servidor e `ArgsValidos`.
- Lento: as respostas do agente (Ollama) e o juiz (Gemma no OpenRouter, precisa de `OPENROUTER_API_KEY`).
- BLEU e ROUGE-L precisam de `pip install sacrebleu rouge_score`.
- O tempo de cada chamada MCP é registado na secção 4. Respostas guardadas por uma versão antiga não o têm (`LatenciaMedia` fica vazia): põe `REGENERAR = True` para as voltar a gerar.

## 1. Setup

In [1]:
import json
import math
import os
import sys
import time
from datetime import datetime

import pandas as pd
from IPython.display import Markdown, display

PASTA_AVALIACAO = next(
    d for d in (os.getcwd(), os.path.join(os.getcwd(), "avaliacao"), os.path.join(os.getcwd(), "mcp", "avaliacao"))
    if os.path.exists(os.path.join(d, "metricas_d3.py"))
)
PASTA_D3 = os.path.dirname(PASTA_AVALIACAO)
PASTA_PROJETO = os.path.dirname(PASTA_D3)
PASTA_D2_AVALIACAO = os.path.join(PASTA_PROJETO, "rag", "evaluation")
sys.path[:0] = [PASTA_AVALIACAO, os.path.join(PASTA_D3, "cliente"), PASTA_D2_AVALIACAO]
os.environ.setdefault("RAGAS_DO_NOT_TRACK", "true")

import metricas_d3 as D3   # métricas do D3
import metrics as M        # avaliador Ragas do D2 (Faithfulness, Answer Relevancy)

In [2]:
from dotenv import load_dotenv

load_dotenv(os.path.join(PASTA_PROJETO, ".env"))   # D2_D3_Miguel_Luís/.env
# Juiz no OpenRouter (endpoint OpenAI-compatível): maior e de outra família que o agente
JUIZ_MODELO = os.environ.get("JUIZ_MODELO", "google/gemma-4-31b-it")
JUIZ_URL = "https://openrouter.ai/api/v1"
JUIZ_CHAVE = os.environ.get("OPENROUTER_API_KEY")
if not JUIZ_CHAVE:
    raise RuntimeError("Falta OPENROUTER_API_KEY (no .env ou no ambiente) para o juiz.")

MODOS = ["rag_primeiro", "livre"]   # modos do agente a avaliar e comparar
CATEGORIAS = None      # ex.: ["web", "batcaverna"]; None = todas (arquivos, web, batcaverna, batman)
LIMITE = None          # número máximo de perguntas; None = todas
REGENERAR = False      # True: ignora as respostas guardadas e volta a correr o agente
REPETICOES = 3         # vezes que cada ferramenta é chamada nos testes diretos

MAX_CONTEXTO = 3000            # caracteres por saída de ferramenta passados ao juiz
N_PERGUNTAS_RELEVANCIA = 3     # N do Answer Relevancy
EMBEDDINGS = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"  # o mesmo do RAG - Multilingual

GT_D2 = os.path.join(PASTA_D2_AVALIACAO, "ground_truth_example.json")
GT_D3 = os.path.join(PASTA_AVALIACAO, "ground_truth_d3.json")
PASTA_RESULTADOS = os.path.join(PASTA_AVALIACAO, "resultados")


def ficheiro_respostas(modo):
    """Cache das respostas do agente num modo."""
    return os.path.join(PASTA_RESULTADOS, f"respostas_{modo}.json")


def guardar_json(caminho, dados):
    os.makedirs(os.path.dirname(caminho), exist_ok=True)
    with open(caminho, "w", encoding="utf-8") as f:
        json.dump(dados, f, ensure_ascii=False, indent=2, default=str)

## 2. Ground truth

As perguntas do D2 entram com a categoria `arquivos` e a ferramenta esperada `pesquisar_arquivos`. 

In [3]:
def carregar_ground_truth():
    """Perguntas do D2 (categoria 'arquivos') + perguntas do D3."""
    with open(GT_D2, encoding="utf-8") as f:
        perguntas_d2 = json.load(f)
    with open(GT_D3, encoding="utf-8") as f:
        perguntas_d3 = json.load(f)

    # As perguntas do D2 não têm os campos novos do D3: acrescentam-se aqui
    for pergunta in perguntas_d2:
        pergunta["categoria"] = "arquivos"
        pergunta["ferramentas_esperadas"] = ["pesquisar_arquivos"]
        pergunta["ferramentas_permitidas"] = []
        pergunta["resposta_variavel"] = False

    itens = perguntas_d2 + perguntas_d3
    ids = [item["query_id"] for item in itens]
    if len(ids) != len(set(ids)):
        raise ValueError("há query_id repetidos")
    return itens


itens = carregar_ground_truth()
if CATEGORIAS:
    itens = [item for item in itens if item["categoria"] in CATEGORIAS]
if LIMITE:
    itens = itens[:LIMITE]

print(f"{len(itens)} perguntas")
pd.DataFrame(itens)[["query_id", "categoria", "user_query", "ferramentas_esperadas", "resposta_variavel"]]

26 perguntas


,query_id,categoria,user_query,ferramentas_esperadas,resposta_variavel
0,1,arquivos,Quem assassinou os pais de Bruce Wayne?,[pesquisar_arquivos],False
1,2,arquivos,Quem fundou o Asilo Arkham e em honra de quem?,[pesquisar_arquivos],False
2,3,arquivos,Qual é o nome verdadeiro da Arlequina?,[pesquisar_arquivos],False
3,4,arquivos,Em que revista e em que ano se estreou o Batman?,[pesquisar_arquivos],False
4,5,arquivos,Quem criou o Batman?,[pesquisar_arquivos],False
5,6,arquivos,Qual é o nome completo do Batman?,[pesquisar_arquivos],False
6,7,arquivos,O que é o Poço de Lázaro?,[pesquisar_arquivos],False
7,8,arquivos,De onde vem o nome Gotham City?,[pesquisar_arquivos],False
8,9,arquivos,Quais são os três moradores mais famosos de Go...,[pesquisar_arquivos],False
9,10,arquivos,De onde foi retirado o nome do Asilo Arkham?,[pesquisar_arquivos],False


## 3. Cada ferramenta MCP sozinha (sem LLM)

Aqui não há agente: o notebook liga-se ao servidor MCP e chama cada ferramenta diretamente, `REPETICOES` vezes, com argumentos fixos (a lista `TESTES` em `metricas_d3.py`). Para cada chamada guarda-se:

- **`respondeu`**: o servidor devolveu uma resposta (sem exceção nem falha de ligação). A média é a **fiabilidade do servidor**: `TesteRespondeu`.
- **`correto`**: o resultado passa a verificação do teste (campos certos, valores plausíveis). Dois testes usam entradas inválidas de propósito e só passam se a ferramenta devolver `erro`. A média é `TesteCorreto`.
- **`segundos`**: a **latência** da chamada.

Também se pede ao servidor o esquema dos argumentos de cada ferramenta (`list_tools`), que a secção 5 usa para conferir os argumentos que o LLM envia.

As ferramentas web precisam de internet; `pesquisar_arquivos` e `relevancia` medem aqui só o formato (a qualidade da recuperação foi avaliada no D2).

In [4]:
import agente_mcp
from fastmcp import Client

ESQUEMAS, DESCRICOES = {}, {}   # {ferramenta: esquema dos argumentos / descrição}
linhas = []                     # uma linha por chamada de teste

async with Client(agente_mcp.SERVIDOR) as cliente:
    # 1. O servidor diz que ferramentas tem e que argumentos cada uma aceita
    for ferramenta in await cliente.list_tools():
        ESQUEMAS[ferramenta.name] = getattr(ferramenta, "input_schema", None) or ferramenta.inputSchema
        DESCRICOES[ferramenta.name] = ferramenta.description or ""

    # 2. Cada teste chama a ferramenta REPETICOES vezes e mede o tempo
    for teste in D3.TESTES:
        for repeticao in range(1, REPETICOES + 1):
            inicio = time.perf_counter()
            try:
                resposta = await cliente.call_tool(teste["ferramenta"], teste["argumentos"])
                resultado = json.loads(agente_mcp.extrair_texto(resposta))
                respondeu = True
            except Exception as e:   # o servidor falhou ou não respondeu
                resultado, respondeu = {"erro": f"{type(e).__name__}: {e}"}, False
            segundos = time.perf_counter() - inicio

            linhas.append({
                "ferramenta": teste["ferramenta"],
                "argumentos": json.dumps(teste["argumentos"], ensure_ascii=False),
                "repeticao": repeticao,
                "respondeu": respondeu,
                "correto": respondeu and D3.verificar(teste, resultado),
                "segundos": round(segundos, 3),
            })

testes_df = pd.DataFrame(linhas)
guardar_json(os.path.join(PASTA_RESULTADOS, "esquemas_ferramentas.json"), ESQUEMAS)
testes_df.to_csv(os.path.join(PASTA_RESULTADOS, "testes_ferramentas.csv"), index=False)

sem_teste = set(ESQUEMAS) - set(testes_df["ferramenta"])
if sem_teste:
    print("Ferramentas sem teste:", sorted(sem_teste))

# Por ferramenta
testes_por_ferramenta = testes_df.groupby("ferramenta").agg(
    chamadas=("correto", "size"),
    TesteRespondeu=("respondeu", "mean"),
    TesteCorreto=("correto", "mean"),
    latencia_media_s=("segundos", "mean"),
    latencia_max_s=("segundos", "max"),
)
display(testes_por_ferramenta.round(3))
print(f"Servidor: respondeu em {100 * testes_df['respondeu'].mean():.1f}% das chamadas, "
      f"resultado correto em {100 * testes_df['correto'].mean():.1f}%.")

# Os testes que falharam
testes_df[~testes_df["correto"]]

,chamadas,TesteRespondeu,TesteCorreto,latencia_media_s,latencia_max_s
ferramenta,,,,,
clima_gotham,3,1.0,1.0,0.001,0.001
estado_batmovel,3,1.0,1.0,0.001,0.001
gerir_agenda_bruce,9,1.0,1.0,0.001,0.001
ler_pagina,6,1.0,1.0,0.220,0.688
listar_arquivos,3,1.0,1.0,0.010,0.024
perguntar_ao_batman,3,1.0,1.0,0.006,0.013
pesquisar_arquivos,3,1.0,1.0,0.027,0.057
pesquisar_noticias,3,1.0,1.0,0.658,0.867
pesquisar_web,3,1.0,1.0,2.484,2.844


Servidor: respondeu em 100.0% das chamadas, resultado correto em 100.0%.


,ferramenta,argumentos,repeticao,respondeu,correto,segundos


## 4. Respostas do agente

Cada pergunta corre sem histórico, em cada modo de `MODOS`. Cada modo tem a sua ligação ao servidor MCP (o servidor arranca de novo, por isso o schedule volta ao estado inicial e os modos não se influenciam). 

As respostas ficam em `resultados/respostas_<modo>.json` e são reutilizadas na execução seguinte (só se geram as que faltam; respostas com erro de uma execução anterior são ignoradas). Uma resposta que falha **não** é guardada: corre-se a célula utra vez.

In [5]:
REGISTO = []   # chamadas MCP da pergunta que está a ser respondida

# Guarda o _chamar original uma só vez, para a célula poder ser corrida de novo sem embrulhar duas vezes
if not hasattr(agente_mcp, "_chamar_original"):
    agente_mcp._chamar_original = agente_mcp._chamar


async def chamar_com_registo(cliente, nome, args, origem, passos):
    inicio = time.perf_counter()
    ok, texto = await agente_mcp._chamar_original(cliente, nome, args, origem, passos)
    REGISTO.append({"ferramenta": nome, "argumentos": args, "origem": origem, "ok": ok,
                    "segundos": round(time.perf_counter() - inicio, 3), "saida": texto})
    return ok, texto


agente_mcp._chamar = chamar_com_registo


async def gerar_respostas(modo, guardadas):
    """Corre o agente MCP, num modo, nas perguntas que ainda não têm resposta guardada."""
    respostas = dict(guardadas)
    por_fazer = [item for item in itens if item["query_id"] not in respostas]
    if not por_fazer:
        return respostas   # tudo guardado: não é preciso arrancar o servidor MCP nem o LLM do agente

    llm = agente_mcp.obter_llm()
    async with Client(agente_mcp.SERVIDOR) as cliente:
        for n, item in enumerate(por_fazer, start=1):
            print(f"[{modo} {n}/{len(por_fazer)}] {item['user_query']}")
            REGISTO.clear()
            try:
                r = await agente_mcp.responder_com_cliente(cliente, item["user_query"], [], modo=modo, llm=llm)
            except Exception as e:
                print(f"  ERRO: {type(e).__name__}: {e}")   # não se guarda: corre-se outra vez
                continue

            respostas[item["query_id"]] = {
                **item,
                "resposta": M.strip_thinking(str(r.get("resposta", ""))),
                "modo": modo,
                "limite_atingido": bool(r.get("limite_atingido", False)),
                "segundos": r.get("segundos"),
                "chamadas": list(REGISTO),
            }
            guardar_json(ficheiro_respostas(modo), list(respostas.values()))   # a cada pergunta, para poder retomar
    return respostas

In [6]:
respostas = {}   # {modo: {query_id: resposta}}

for modo in MODOS:
    guardadas = {}
    if os.path.exists(ficheiro_respostas(modo)) and not REGENERAR:
        with open(ficheiro_respostas(modo), encoding="utf-8") as f:
            guardadas = {s["query_id"]: s for s in json.load(f) if not s.get("erro")}
        print(f"{modo}: {len(guardadas)} respostas reutilizadas")

    # Top-level await: o Jupyter já tem um event loop a correr
    todas = await gerar_respostas(modo, guardadas)

    faltam = [item["query_id"] for item in itens if item["query_id"] not in todas]
    if faltam:
        raise RuntimeError(f"{modo}: faltam as respostas {faltam}. O Ollama está ligado? Corre esta célula outra vez.")
    respostas[modo] = {item["query_id"]: todas[item["query_id"]] for item in itens}   # pela ordem do ground truth

# Ferramentas usadas por cada modo, lado a lado
linhas = []
for item in itens:
    linha = {"query_id": item["query_id"], "categoria": item["categoria"]}
    for modo in MODOS:
        chamadas = respostas[modo][item["query_id"]]["chamadas"]
        linha[modo] = ", ".join(f"{c['ferramenta']} ({c['origem']})" for c in chamadas)
    linhas.append(linha)
pd.DataFrame(linhas).set_index("query_id")

rag_primeiro: 0 respostas reutilizadas
[rag_primeiro 1/26] Quem assassinou os pais de Bruce Wayne?
[rag_primeiro 2/26] Quem fundou o Asilo Arkham e em honra de quem?
[rag_primeiro 3/26] Qual é o nome verdadeiro da Arlequina?
[rag_primeiro 4/26] Em que revista e em que ano se estreou o Batman?
[rag_primeiro 5/26] Quem criou o Batman?
[rag_primeiro 6/26] Qual é o nome completo do Batman?
[rag_primeiro 7/26] O que é o Poço de Lázaro?
[rag_primeiro 8/26] De onde vem o nome Gotham City?
[rag_primeiro 9/26] Quais são os três moradores mais famosos de Gotham City?
[rag_primeiro 10/26] De onde foi retirado o nome do Asilo Arkham?
[rag_primeiro 11/26] Em que revista apareceu pela primeira vez o Asilo Arkham?
[rag_primeiro 12/26] Onde fica a Batcaverna?
[rag_primeiro 13/26] Como é que Bruce Wayne descobriu as cavernas quando era criança?
[rag_primeiro 14/26] Por quanto foi vendido o Batmóvel da série de televisão dos anos sessenta?
[rag_primeiro 15/26] A partir de que carro foi construído o Batm

,categoria,rag_primeiro,livre
query_id,,,
1,arquivos,"relevancia (codigo), pesquisar_arquivos (codigo)",relevancia (codigo)
2,arquivos,"relevancia (codigo), pesquisar_arquivos (codigo)","relevancia (codigo), pesquisar_arquivos (llm)"
3,arquivos,"relevancia (codigo), pesquisar_arquivos (codigo)",relevancia (codigo)
4,arquivos,"relevancia (codigo), pesquisar_arquivos (codigo)","relevancia (codigo), pesquisar_arquivos (llm),..."
5,arquivos,"relevancia (codigo), pesquisar_arquivos (codigo)",relevancia (codigo)
6,arquivos,"relevancia (codigo), pesquisar_arquivos (codigo)",relevancia (codigo)
7,arquivos,"relevancia (codigo), pesquisar_arquivos (codigo)",relevancia (codigo)
8,arquivos,"relevancia (codigo), pesquisar_arquivos (codigo)",relevancia (codigo)
9,arquivos,"relevancia (codigo), pesquisar_arquivos (codigo)","relevancia (codigo), pesquisar_web (llm)"


## 5. Métricas

### 5.1 Ferramentas, servidor e argumentos válidos (sem LLM)

- **Ferramentas:** ToolHit, ToolRecall, ToolPrecision e NumToolCalls. A precisão só conta as ferramentas pedidas pelo LLM (as chamadas automáticas do código não são uma escolha); o recall conta todas.
- **Servidor:** `TaxaSucesso` (chamadas MCP sem erro) e `LatenciaMedia` (segundos por chamada), a partir das chamadas registadas.


Além da tabela por pergunta, faz-se uma **tabela por chamada** (`chamadas_df`, uma linha por chamada MCP), que serve para as médias por ferramenta.

In [7]:
def metricas_da_pergunta(s):
    linha = {"query_id": s["query_id"], "categoria": s["categoria"],
             "limite_atingido": s["limite_atingido"], "segundos": s["segundos"]}
    linha.update(D3.metricas_ferramentas(s["chamadas"], s["ferramentas_esperadas"], s["ferramentas_permitidas"]))
    linha.update(D3.metricas_servidor(s["chamadas"]))
    return linha


# Uma linha por pergunta, em cada modo
base = {modo: pd.DataFrame([metricas_da_pergunta(s) for s in respostas[modo].values()]) for modo in MODOS}

# Uma linha por chamada MCP, em cada modo. Os argumentos só se avaliam nas chamadas pedidas pelo LLM
linhas = []
for modo in MODOS:
    for s in respostas[modo].values():
        for c in s["chamadas"]:
            pedida_pelo_llm = c["origem"] == "llm"
            linhas.append({
                "modo": modo,
                "query_id": s["query_id"],
                "pergunta": s["user_query"],
                "ferramenta": c["ferramenta"],
                "argumentos": c["argumentos"],
                "origem": c["origem"],
                "ok": c["ok"],
                "segundos": c.get("segundos", math.nan),
                "ArgsValidos": D3.argumentos_validos(c["argumentos"], ESQUEMAS.get(c["ferramenta"], {})) if pedida_pelo_llm else math.nan,
                "ArgsAdequados": math.nan,   # preenchido na secção 5.2
            })
chamadas_df = pd.DataFrame(linhas)

# Médias por categoria, um bloco por modo
pd.concat({modo: base[modo].drop(columns=["query_id"]).groupby("categoria").mean(numeric_only=True) for modo in MODOS}).round(3)

limite_atingido  segundos  ToolHit  ToolRecall  \
             categoria                                                    
rag_primeiro arquivos                0.0     5.971    1.000       1.000   
             batcaverna              0.0     6.980    1.000       1.000   
             batman                  0.0     8.010    1.000       1.000   
             web                     0.0     5.094    0.000       0.000   
livre        arquivos                0.0     5.495    0.312       0.312   
             batcaverna              0.0     3.912    1.000       1.000   
             batman                  0.0     3.720    1.000       1.000   
             web                     0.0    10.472    1.000       1.000   

                         ToolPrecision  NumToolCalls  TaxaSucesso  \
             categoria                                              
rag_primeiro arquivos            0.500         1.125        1.000   
             batcaverna          1.000         2.000        1.000   
             batman              1.000         2.000        1.000   
             web                   NaN         1.000        1.000   
livre        arquivos            0.562         0.562        0.885   
             batcaverna          1.000         1.000        1.000   
             batman              1.000         1.000        1.000   
             web                 0.833         1.800        0.767   

                         LatenciaMedia  
             categoria                  
rag_primeiro arquivos            0.026  
             batcaverna          0.017  
             batman              0.023  
             web                 0.024  
livre        arquivos            0.254  
             batcaverna          0.020  
             batman              0.026  
             web                 0.909

### 5.2 Juiz Gemma: correção, resposta, fundamentação e argumentos adequados

- **`correctness`** (pass/fail): `DiscreteMetric` do Ragas com o prompt do guia. O prompt do guia pede "factually accurate based on the provided context" mas não passa o contexto; aqui o contexto são as saídas das ferramentas MCP.
- **`JudgeScore`**: "total rating" de 0 a 10 com o prompt dos slides (só pergunta e resposta, sem referência).
- **`Faithfulness`**: só quando houve ferramentas com saída. **`AnswerRelevancy`**: sempre.
- **BLEU / ROUGE-L**: só quando a resposta esperada é um facto fixo.

In [8]:
from openai import AsyncOpenAI
from ragas.llms import llm_factory

if JUIZ_MODELO == os.environ.get("AGENTE_MODELO", "qwen2.5:7b"):
    print(f"Aviso: o juiz ({JUIZ_MODELO}) é o mesmo modelo do agente; muda JUIZ_MODELO para outro.")

cliente_juiz = AsyncOpenAI(base_url=JUIZ_URL, api_key=JUIZ_CHAVE)
llm_ragas = llm_factory(JUIZ_MODELO, provider="openai", client=cliente_juiz, temperature=0, max_tokens=4096)
correcao = D3.criar_metrica_correcao()
avaliador = M.RagasEvaluator(JUIZ_MODELO, EMBEDDINGS, N_PERGUNTAS_RELEVANCIA,
                             base_url=JUIZ_URL, api_key=JUIZ_CHAVE, reasoning_effort=None)

# BLEU / ROUGE-L do Ragas (precisam de sacrebleu e rouge_score)
try:
    import sacrebleu  # noqa: F401
    import rouge_score  # noqa: F401
    from ragas.metrics.collections import BleuScore, RougeScore
    bleu, rouge = BleuScore(), RougeScore(rouge_type="rougeL")
except ImportError:
    bleu = rouge = None
    print("Aviso: BLEU/ROUGE desligados (pip install sacrebleu rouge_score)")

/Users/miguelmauricioantonio/Documents/MIA/2/ILN/PROJECT/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 14121.28it/s]


In [9]:
async def avaliar_resposta(s):
    """Métricas com juiz de uma pergunta: correção, nota 0-10, Faithfulness, AnswerRelevancy, BLEU, ROUGE-L."""
    resposta = s["resposta"]
    # O contexto são as saídas das ferramentas que correram bem
    contextos = [c["saida"][:MAX_CONTEXTO] for c in s["chamadas"]
                 if c["ok"] and c["ferramenta"] not in D3.FERRAMENTAS_IGNORADAS]
    r = {"query_id": s["query_id"], "model_response": resposta}

    try:
        score = await correcao.ascore(
            question=s["user_query"], expected_answer=s["ground_truth_answer"],
            context="\n\n".join(contextos) or "(no tool was called)",
            response=resposta, llm=llm_ragas)
        r["correctness_score"], r["correctness_reason"] = score.value, score.reason
    except Exception as e:
        r["correctness_score"], r["correctness_reason"] = None, f"ERRO: {type(e).__name__}: {e}"

    try:
        r["JudgeScore"], r["judge_feedback"] = await D3.nota_llm_judge(
            cliente_juiz, JUIZ_MODELO, s["user_query"], resposta)
    except Exception as e:
        r["JudgeScore"], r["judge_feedback"] = math.nan, f"ERRO: {type(e).__name__}: {e}"

    # Faithfulness só faz sentido se houver contexto (saídas das ferramentas)
    so = ["AnswerRelevance"] + (["Faithfulness"] if contextos else [])
    scores, erros = await avaliador.ascore(s["user_query"], s["ground_truth_answer"], resposta, contextos, only=so)
    r["Faithfulness"] = scores.get("Faithfulness", math.nan)
    r["AnswerRelevancy"] = scores.get("AnswerRelevance", math.nan)
    r["ragas_erros"] = "; ".join(f"{k}: {v}" for k, v in erros.items())

    # BLEU / ROUGE-L só para respostas fixas
    r["BLEU"] = r["ROUGE-L"] = math.nan
    if bleu and not s["resposta_variavel"]:
        r["BLEU"] = (await bleu.ascore(reference=s["ground_truth_answer"], response=resposta)).value
        r["ROUGE-L"] = (await rouge.ascore(reference=s["ground_truth_answer"], response=resposta)).value
    return r


# 1. Uma avaliação por pergunta, em cada modo
linhas_juiz = {}
for modo in MODOS:
    linhas_juiz[modo] = []
    for n, s in enumerate(respostas[modo].values(), start=1):
        print(f"[{modo} {n}/{len(itens)}] {s['user_query']}")
        linhas_juiz[modo].append(await avaliar_resposta(s))

# 2. Argumentos adequados: uma avaliação por chamada pedida pelo LLM
for i, c in chamadas_df[chamadas_df["origem"] == "llm"].iterrows():
    try:
        nota, _ = await D3.argumentos_adequados(
            cliente_juiz, JUIZ_MODELO, c["pergunta"], c["ferramenta"], DESCRICOES.get(c["ferramenta"], ""), c["argumentos"])
    except Exception as e:
        print(f"  ERRO nos argumentos de {c['ferramenta']}: {type(e).__name__}: {e}")
        nota = math.nan
    chamadas_df.loc[i, "ArgsAdequados"] = nota

[rag_primeiro 1/26] Quem assassinou os pais de Bruce Wayne?
[rag_primeiro 2/26] Quem fundou o Asilo Arkham e em honra de quem?
[rag_primeiro 3/26] Qual é o nome verdadeiro da Arlequina?
[rag_primeiro 4/26] Em que revista e em que ano se estreou o Batman?
[rag_primeiro 5/26] Quem criou o Batman?
[rag_primeiro 6/26] Qual é o nome completo do Batman?
[rag_primeiro 7/26] O que é o Poço de Lázaro?
[rag_primeiro 8/26] De onde vem o nome Gotham City?
[rag_primeiro 9/26] Quais são os três moradores mais famosos de Gotham City?
[rag_primeiro 10/26] De onde foi retirado o nome do Asilo Arkham?
[rag_primeiro 11/26] Em que revista apareceu pela primeira vez o Asilo Arkham?
[rag_primeiro 12/26] Onde fica a Batcaverna?
[rag_primeiro 13/26] Como é que Bruce Wayne descobriu as cavernas quando era criança?
[rag_primeiro 14/26] Por quanto foi vendido o Batmóvel da série de televisão dos anos sessenta?
[rag_primeiro 15/26] A partir de que carro foi construído o Batmóvel da série de televisão de 1966?
[ra

In [10]:
tabelas = {}   # {modo: uma linha por pergunta, com todas as métricas}
for modo in MODOS:
    juiz_df = pd.DataFrame(linhas_juiz[modo])
    # Argumentos por pergunta = média das chamadas dessa pergunta
    argumentos_df = (chamadas_df[chamadas_df["modo"] == modo]
                     .groupby("query_id")[["ArgsValidos", "ArgsAdequados"]].mean().reset_index())
    tabela = base[modo].merge(juiz_df, on="query_id", how="left").merge(argumentos_df, on="query_id", how="left")

    # Causa de cada resposta errada (só para correctness = fail)
    falhou = tabela["correctness_score"] == "fail"
    tabela["causa_falha"] = [D3.classificar_falha(linha) if f else None
                             for linha, f in zip(tabela.to_dict("records"), falhou)]
    tabelas[modo] = tabela

# Correctness, nota do juiz e causa da falha por pergunta, nos dois modos lado a lado
pd.concat({modo: tabelas[modo].set_index("query_id")[["correctness_score", "JudgeScore", "causa_falha"]]
           for modo in MODOS}, axis=1)

rag_primeiro                                            livre  \
         correctness_score JudgeScore         causa_falha correctness_score   
query_id                                                                      
1                     pass       10.0                 NaN              fail   
2                     pass       10.0                 NaN              fail   
3                     pass       10.0                 NaN              fail   
4                     pass       10.0                 NaN              pass   
5                     pass       10.0                 NaN              pass   
6                     pass       10.0                 NaN              pass   
7                     pass       10.0                 NaN              fail   
8                     pass       10.0                 NaN              fail   
9                     pass        5.0                 NaN              pass   
10                    pass       10.0                 NaN              fail   
11                    pass       10.0                 NaN              fail   
12                    pass       10.0                 NaN              pass   
13                    pass        4.0                 NaN              fail   
14                    pass       10.0                 NaN              pass   
15                    pass       10.0                 NaN              fail   
101                   pass       10.0                 NaN              pass   
102                   pass       10.0                 NaN              pass   
103                   pass       10.0                 NaN              pass   
104                   pass       10.0                 NaN              pass   
105                   fail        2.0  selecao_ferramenta              pass   
201                   pass       10.0                 NaN              pass   
202                   pass       10.0                 NaN              pass   
203                   pass       10.0                 NaN              pass   
204                   pass       10.0                 NaN              pass   
205                   fail       10.0             geracao              pass   
206                   pass       10.0                 NaN              pass   

                                         
         JudgeScore         causa_falha  
query_id                                 
1               4.0  selecao_ferramenta  
2               8.0   ferramenta_falhou  
3               2.0  selecao_ferramenta  
4              10.0                 NaN  
5              10.0                 NaN  
6              10.0                 NaN  
7               2.0  selecao_ferramenta  
8               7.0  selecao_ferramenta  
9               5.0                 NaN  
10              1.0   ferramenta_falhou  
11              0.0   ferramenta_falhou  
12             10.0                 NaN  
13             10.0  selecao_ferramenta  
14             10.0                 NaN  
15             10.0  selecao_ferramenta  
101            10.0                 NaN  
102            10.0                 NaN  
103            10.0                 NaN  
104            10.0                 NaN  
105             1.0                 NaN  
201            10.0                 NaN  
202            10.0                 NaN  
203            10.0                 NaN  
204            10.0                 NaN  
205             2.0                 NaN  
206            10.0                 NaN

## 6. Resumo e análise de falhas

Guarda em `resultados/<data-hora>/`: `testes_ferramentas.csv`, e em cada modo `por_pergunta.csv`, `por_chamada.csv`, `resumo.json` e `analise_falhas.md`.

A análise de falhas é ao estilo da secção [*Analysis of actual failure patterns*](https://docs.ragas.io/en/stable/howtos/applications/evaluate-and-improve-rag/#analysis-of-actual-failure-patterns-from-our-evaluation) do guia: cada resposta com `correctness = fail` recebe **a primeira** destas causas que se aplicar:

1. `selecao_ferramenta`: o LLM não chamou a ferramenta esperada (ToolRecall < 1);
2. `ferramenta_falhou`: uma chamada MCP devolveu erro, com argumentos bons (culpa do servidor);
3. `limite_chamadas`: o agente atingiu o limite de 4 chamadas;
4. `geracao`: tudo correu bem e a resposta saiu errada.

In [11]:
PASTA_EXECUCAO = os.path.join(PASTA_RESULTADOS, datetime.now().strftime("%Y%m%d-%H%M%S"))
os.makedirs(PASTA_EXECUCAO, exist_ok=True)
testes_df.to_csv(os.path.join(PASTA_EXECUCAO, "testes_ferramentas.csv"), index=False)

for modo in MODOS:
    pasta = os.path.join(PASTA_EXECUCAO, modo)
    os.makedirs(pasta, exist_ok=True)
    tabela = tabelas[modo]
    tabela.to_csv(os.path.join(pasta, "por_pergunta.csv"), index=False)
    chamadas_df[chamadas_df["modo"] == modo].to_csv(os.path.join(pasta, "por_chamada.csv"), index=False)

    passou = tabela["correctness_score"].eq("pass")
    guardar_json(os.path.join(pasta, "resumo.json"), {
        "modo": modo,
        "juiz": JUIZ_MODELO,
        "n_perguntas": len(tabela),
        "taxa_pass": round(float(passou.mean()), 4),
        "taxa_pass_por_categoria": passou.groupby(tabela["categoria"]).mean().round(4).to_dict(),
        "medias": tabela.drop(columns=["query_id"]).mean(numeric_only=True).round(4).to_dict(),
        "falhas_por_causa": tabela["causa_falha"].value_counts().to_dict(),
    })
    print(f"{modo}: correctness {100 * passou.mean():.1f}% pass")
print(f"Resultados em {PASTA_EXECUCAO}")

rag_primeiro: correctness 92.3% pass
livre: correctness 65.4% pass
Resultados em /Users/miguelmauricioantonio/Documents/MIA/2/ILN/PROJECT/D2_D3_Miguel_Luís/mcp/avaliacao/resultados/20261004-120944


In [12]:
def analise_falhas(tabela, modo):
    """Relatório em Markdown das respostas com correctness = fail."""
    falhas = tabela[tabela["correctness_score"] == "fail"]
    passou = int(tabela["correctness_score"].eq("pass").sum())
    texto = [
        f"# Análise de falhas do D3 (modo `{modo}`)", "",
        f"Juiz: `{JUIZ_MODELO}`. Correctness: **{passou}/{len(tabela)} pass** ({100 * passou / len(tabela):.1f}%). "
        f"JudgeScore médio: {tabela['JudgeScore'].mean():.2f}/10.", "",
        "## Falhas por causa", "", "| Causa | Significado | N |", "|---|---|---|",
    ]
    for causa, n in falhas["causa_falha"].value_counts().items():
        texto.append(f"| {causa} | {D3.CAUSAS[causa]} | {n} |")

    texto += ["", "## Casos", ""]
    for _, f in falhas.iterrows():
        s = respostas[modo][int(f["query_id"])]
        usadas = ", ".join(f"{c['ferramenta']} ({c['origem']}{'' if c['ok'] else ', erro'}, {c.get('segundos', '?')}s)"
                           for c in s["chamadas"]) or "nenhuma"
        texto += [
            f"### [{f['query_id']}] {s['user_query']}", "",
            f"- **Categoria:** {s['categoria']}  ",
            f"- **Causa provável:** {f['causa_falha']} ({D3.CAUSAS[f['causa_falha']]})  ",
            f"- **Ferramentas esperadas:** {', '.join(s['ferramentas_esperadas'])}  ",
            f"- **Ferramentas chamadas:** {usadas}  ",
            f"- **Resposta esperada:** {s['ground_truth_answer']}  ",
            f"- **Resposta do Alfred:** {s['resposta'][:600]}{'…' if len(s['resposta']) > 600 else ''}  ",
            f"- **Razão do juiz:** {f['correctness_reason']}", "",
        ]
    return "\n".join(texto)


for modo in MODOS:
    relatorio = analise_falhas(tabelas[modo], modo)
    with open(os.path.join(PASTA_EXECUCAO, modo, "analise_falhas.md"), "w", encoding="utf-8") as f:
        f.write(relatorio)
    display(Markdown(relatorio))

# Análise de falhas do D3 (modo `rag_primeiro`)

Juiz: `google/gemma-4-31b-it`. Correctness: **24/26 pass** (92.3%). JudgeScore médio: 9.27/10.

## Falhas por causa

| Causa | Significado | N |
|---|---|---|
| selecao_ferramenta | o LLM não chamou a ferramenta esperada | 1 |
| geracao | chamou as ferramentas certas, sem erros, mas a resposta saiu errada | 1 |

## Casos

### [105] Quais são as notícias mais recentes sobre o próximo filme do Batman?

- **Categoria:** web  
- **Causa provável:** selecao_ferramenta (o LLM não chamou a ferramenta esperada)  
- **Ferramentas esperadas:** pesquisar_noticias|pesquisar_web  
- **Ferramentas chamadas:** relevancia (codigo, 0.035s), pesquisar_arquivos (codigo, 0.012s)  
- **Resposta esperada:** Resume notícias recentes sobre o próximo filme do Batman, com fonte ou data, obtidas com as ferramentas de pesquisa e sem inventar factos.  
- **Resposta do Alfred:** Não tenho acesso a notícias recentes sobre o próximo filme do Batman diretamente na minha base de dados local. No entanto, posso verificar as últimas informações disponíveis em meu banco de dados.

De acordo com os arquivos locais do Batcomputador, o filme "The Batman" teve seu lançamento adiado para março de 2022, mas foi adiado novamente. Não há informações mais recentes sobre novos lançamentos ou detalhes do filme.

Se quiser mais detalhes, posso pesquisar na web para você. Gostaria que eu fizesse isso?  
- **Razão do juiz:** The model failed to properly utilize the provided context. The context contained specific information: Robert Pattinson's confirmation as Batman (May 2019), the delay of 'The Batman' to March 2022 (October 2020), and information about Batman's appearances in 'The Flash' (released June 2023). The model claimed it had no access to recent news and provided a vague answer, ignoring the specific details available in the tool outputs.

### [205] Pergunta ao Batman o que ele pensa do Coringa.

- **Categoria:** batman  
- **Causa provável:** geracao (chamou as ferramentas certas, sem erros, mas a resposta saiu errada)  
- **Ferramentas esperadas:** perguntar_ao_batman  
- **Ferramentas chamadas:** relevancia (codigo, 0.044s), pesquisar_arquivos (codigo, 0.011s), perguntar_ao_batman (llm, 0.015s)  
- **Resposta esperada:** Transmite a resposta do Batman obtida com a ferramenta perguntar_ao_batman, sem a inventar.  
- **Resposta do Alfred:** Senhor, o Coringa é um indivíduo perigoso e psicopata, sempre em busca de diversão ao custo de vidas inocentes. Ele é uma peça crucial em muitas das minhas investigações, embora suas ações sejam imprevisíveis. O que sabes sobre essa pessoa?  
- **Razão do juiz:** The model response hallucinated a detailed description of the Joker ('perigoso e psicopata', 'busca de diversão ao custo de vidas inocentes') that was not present in the tool output. The expected answer explicitly stated that the response should transmit the answer obtained from the tool without inventing it. The tool output was: 'Coringa. Guarda esse nome pode ser uma peça no que estou a investigar. O que sabes sobre essa pessoa?'


# Análise de falhas do D3 (modo `livre`)

Juiz: `google/gemma-4-31b-it`. Correctness: **17/26 pass** (65.4%). JudgeScore médio: 7.38/10.

## Falhas por causa

| Causa | Significado | N |
|---|---|---|
| selecao_ferramenta | o LLM não chamou a ferramenta esperada | 6 |
| ferramenta_falhou | uma chamada MCP devolveu erro (com argumentos bons) | 3 |

## Casos

### [1] Quem assassinou os pais de Bruce Wayne?

- **Categoria:** arquivos  
- **Causa provável:** selecao_ferramenta (o LLM não chamou a ferramenta esperada)  
- **Ferramentas esperadas:** pesquisar_arquivos  
- **Ferramentas chamadas:** relevancia (codigo, 0.029s)  
- **Resposta esperada:** Os pais de Bruce Wayne foram assassinados por Joe Chill.  
- **Resposta do Alfred:** Senhora, essa é uma questão delicada. De acordo com os arquivos do Batcomputador, os pais de Bruce Wayne foram assassinados por um misterioso ladrão durante uma noite de gala em Gotham. O caso permaneceu sem solução por muitos anos, até que Bruce Wayne se tornou o Batman. Para obter mais detalhes, posso pesquisar mais profundamente nos arquivos.  
- **Razão do juiz:** The model response fails to identify the specific individual responsible for the murders (Joe Chill), as required by the expected answer. Instead, it vaguely describes the killer as a 'mysterious thief' and claims the case remained unsolved for years, which contradicts the specific factual answer provided.

### [2] Quem fundou o Asilo Arkham e em honra de quem?

- **Categoria:** arquivos  
- **Causa provável:** ferramenta_falhou (uma chamada MCP devolveu erro (com argumentos bons))  
- **Ferramentas esperadas:** pesquisar_arquivos  
- **Ferramentas chamadas:** relevancia (codigo, 0.038s), pesquisar_arquivos (llm, erro, 0.005s)  
- **Resposta esperada:** O Asilo Arkham foi fundado pelo psiquiatra Amadeus Arkham em honra da sua falecida mãe, Elizabeth Arkham.  
- **Resposta do Alfred:** Peço desculpa, parece que houve um pequeno contratempos na pesquisa dos arquivos. Vamos tentar outra abordagem. De acordo com as minhas memórias, o Asilo Arkham foi fundado em honra ao Dr. Thomas Arkham, um renomado psiquiatra de Gotham que dedicou a sua vida à compreensão e tratamento de doenças mentais. No entanto, a fundação do asilo também está ligada a eventos sombrios que envolveram a família Wayne.  
- **Razão do juiz:** The model response provides incorrect information. It claims the asylum was founded in honor of 'Dr. Thomas Arkham', whereas the expected answer (and established lore) states it was founded by Amadeus Arkham in honor of his mother, Elizabeth Arkham.

### [3] Qual é o nome verdadeiro da Arlequina?

- **Categoria:** arquivos  
- **Causa provável:** selecao_ferramenta (o LLM não chamou a ferramenta esperada)  
- **Ferramentas esperadas:** pesquisar_arquivos  
- **Ferramentas chamadas:** relevancia (codigo, 0.039s)  
- **Resposta esperada:** Harleen Quinzel.  
- **Resposta do Alfred:** A Arlequina verdadeiramente se chama Harley Quinn. É uma ex-médica veterinária que se tornou uma criminosa e aliada de Joker.  
- **Razão do juiz:** The model response identifies the character's name as 'Harley Quinn', which is her alias/superhero name, whereas the expected answer is her real name, 'Harleen Quinzel'. Additionally, the model incorrectly states she was a veterinarian; she was a psychiatrist.

### [7] O que é o Poço de Lázaro?

- **Categoria:** arquivos  
- **Causa provável:** selecao_ferramenta (o LLM não chamou a ferramenta esperada)  
- **Ferramentas esperadas:** pesquisar_arquivos  
- **Ferramentas chamadas:** relevancia (codigo, 0.017s)  
- **Resposta esperada:** O Poço de Lázaro é uma mistura química de poções e venenos tóxicos e místicos que consegue curar corpos doentes ou feridos e ressuscitar os mortos; é por isso que Ra's al Ghul tem idade sobre-humana.  
- **Resposta do Alfred:** O Poço de Lázaro é um local mítico em Gotham, frequentemente mencionado nas histórias do Batman. É supostamente um esconderijo secreto, talvez uma caverna ou uma instalação subterrânea, onde o vilão pode esconder armas, drogas ou outros objetos perigosos. Algumas histórias sugerem que é um local de grande importância para o crime organizado em Gotham. No entanto, não há registros concretos ou provas reais deste local na realidade.  
- **Razão do juiz:** The model response is factually incorrect. It describes the Lazarus Pit as a physical hideout or secret location in Gotham used for storing weapons and drugs, whereas the expected answer correctly identifies it as a chemical mixture of potions and poisons used for healing and resurrection, explaining Ra's al Ghul's longevity.

### [8] De onde vem o nome Gotham City?

- **Categoria:** arquivos  
- **Causa provável:** selecao_ferramenta (o LLM não chamou a ferramenta esperada)  
- **Ferramentas esperadas:** pesquisar_arquivos  
- **Ferramentas chamadas:** relevancia (codigo, 0.037s)  
- **Resposta esperada:** Bill Finger viu o nome "Gotham Jewelers" ao folhear a lista telefónica e escolheu Gotham City, depois de ter pensado em Civic City, Capital City e Coast City.  
- **Resposta do Alfred:** Senhora, o nome "Gotham" tem origens literárias. Foi usado pela primeira vez por Washington Irving em 1839 no seu romance "A Legend of Sleepy Hollow". A cidade foi nomeada em referência a Gotham, um vilarejo fictício em Inglaterra, conhecido por seus moradores que supostamente tentaram mudar o rebanho de cabras em elefantes. No universo do Batman, Gotham City é frequentemente associada a crimes e corrupção, tornando-se um lugar perfeito para o Cavaleiro das Trevas operar.  
- **Razão do juiz:** The model response provides a general literary history of the word 'Gotham' and mentions Washington Irving, but it completely fails to provide the specific origin of the name 'Gotham City' as it relates to the creation of the Batman comics (Bill Finger seeing 'Gotham Jewelers' in a phone book), which is the key information required by the expected answer.

### [10] De onde foi retirado o nome do Asilo Arkham?

- **Categoria:** arquivos  
- **Causa provável:** ferramenta_falhou (uma chamada MCP devolveu erro (com argumentos bons))  
- **Ferramentas esperadas:** pesquisar_arquivos  
- **Ferramentas chamadas:** relevancia (codigo, 0.038s), pesquisar_arquivos (llm, erro, 0.005s)  
- **Resposta esperada:** O nome foi retirado do sanatório da cidade fictícia de Arkham, Massachusetts, descrita nos contos de H.P. Lovecraft.  
- **Resposta do Alfred:** Peço desculpa, parece que houve um pequeno contratempo na pesquisa dos arquivos. Vamos tentar outra abordagem. De acordo com as minhas recordações, o nome "Arkham" foi inspirado no psiquiatra americano Daniel Nathan Arham, que teve uma carreira notória no início do século XX. No universo do Batman, o asilo foi nomeado em homenagem a este personagem fictício, que era supostamente um especialista em psiquiatria e psicologia criminal.  
- **Razão do juiz:** The model response provides completely incorrect information. It claims the name was inspired by a psychiatrist named Daniel Nathan Arham, whereas the expected answer (and factual lore) states it was derived from the fictional city of Arkham, Massachusetts, created by H.P. Lovecraft.

### [11] Em que revista apareceu pela primeira vez o Asilo Arkham?

- **Categoria:** arquivos  
- **Causa provável:** ferramenta_falhou (uma chamada MCP devolveu erro (com argumentos bons))  
- **Ferramentas esperadas:** pesquisar_arquivos  
- **Ferramentas chamadas:** relevancia (codigo, 0.036s), pesquisar_arquivos (llm, erro, 0.004s)  
- **Resposta esperada:** Em Batman #258, de outubro de 1974, escrita por Dennis O'Neil com desenhos de Irv Novick.  
- **Resposta do Alfred:** Peço desculpa, parece que houve um pequeno erro na pesquisa. Vamos tentar de outra forma. Pesquisarei na web para encontrar a informação sobre a primeira aparição do Asilo Arkham em uma revista.  
- **Razão do juiz:** The model failed to provide the answer to the question. Instead of stating that Arkham Asylum first appeared in Batman #258, it apologized for a search error and stated it would try to search the web, providing no factual information.

### [13] Como é que Bruce Wayne descobriu as cavernas quando era criança?

- **Categoria:** arquivos  
- **Causa provável:** selecao_ferramenta (o LLM não chamou a ferramenta esperada)  
- **Ferramentas esperadas:** pesquisar_arquivos  
- **Ferramentas chamadas:** relevancia (codigo, 0.038s)  
- **Resposta esperada:** Bruce Wayne descobriu as cavernas quando caiu através de um poço dilapidado na sua propriedade.  
- **Resposta do Alfred:** Senhora, essa é uma história interessante. Segundo os arquivos do Batcomputador, Bruce Wayne descobriu as cavernas de Gotham quando era criança, por acaso. Estava a brincar com um amigo perto do rio quando este caiu em uma abertura escura. Bruce, corajoso como é, decidiu resgatá-lo e acabou por descobrir as vastas cavernas subterrâneas. Foi um momento que moldou sua personalidade e seu futuro como o Batman.

Se quiser mais detalhes, posso pesquisar mais sobre o assunto.  
- **Razão do juiz:** The model response provides a different narrative than the expected answer. The expected answer states that Bruce fell through a dilapidated well on his property, whereas the model response claims he discovered the caves while trying to rescue a friend who fell into a dark opening near a river. This is a factual discrepancy regarding the specific event.

### [15] A partir de que carro foi construído o Batmóvel da série de televisão de 1966?

- **Categoria:** arquivos  
- **Causa provável:** selecao_ferramenta (o LLM não chamou a ferramenta esperada)  
- **Ferramentas esperadas:** pesquisar_arquivos  
- **Ferramentas chamadas:** relevancia (codigo, 0.035s), ler_pagina (llm, 0.65s)  
- **Resposta esperada:** Foi construído por George Barris a partir de um Lincoln Futura.  
- **Resposta do Alfred:** Senhora, o Batmóvel da série de televisão de 1966 foi baseado no Ford Galaxie. O veículo foi construído por um fabricante de carrocerias chamado John W. Lujan, que transformou um Ford Galaxie 500 em um carro de luxo com uma carroçaria de fibra de vidro. Este carro foi o protótipo do que viria a ser o famoso Batmóvel da série.  
- **Razão do juiz:** The model response claims the 1966 Batmobile was based on a Ford Galaxie and built by John W. Lujan. However, the expected answer states it was built by George Barris from a Lincoln Futura. The model's information is factually incorrect regarding the specific vehicle used for the 1966 series.


## 7. Comparação `rag_primeiro` vs `livre`

As mesmas perguntas nos dois modos, por isso a comparação é emparelhada (pergunta a pergunta), como a comparação "com vs sem RAG" do D2:

- **Médias** de cada métrica nos dois modos e a diferença (`livre` − `rag_primeiro`).
- **Taxa de pass** por categoria.
- **Métricas contínuas** (JudgeScore, AnswerRelevancy, Faithfulness, LatenciaMedia): vitórias / empates / derrotas do `livre` e teste de Wilcoxon.
- **Correctness** (pass/fail): tabela de McNemar (perguntas em que só um dos modos passa) e teste binomial exato sobre esses casos.

O que vale a pena olhar: `ToolRecall` nas perguntas `arquivos` (no `livre` o LLM pode não ir aos arquivos), `ToolPrecision` / `NumToolCalls`  (quantas ferramentas a mais o LLM pede e se as usa bem) e `LatenciaMedia` / `segundos`. Com 26 perguntas os testes têm pouca potência: um p alto não prova que os modos são iguais.

Guarda `comparacao.csv` e `comparacao.json` em `resultados/<data-hora>/`.

In [13]:
from scipy.stats import binomtest, wilcoxon

A, B = MODOS[0], MODOS[1]   # B − A: positivo quer dizer que o segundo modo (livre) é maior
ta = tabelas[A].set_index("query_id")
tb = tabelas[B].set_index("query_id")

METRICAS = ["ToolHit", "ToolRecall", "ToolPrecision", "NumToolCalls", "ArgsValidos", "ArgsAdequados",
            "TaxaSucesso", "LatenciaMedia", "JudgeScore", "AnswerRelevancy", "Faithfulness", "BLEU", "ROUGE-L", "segundos"]
metricas = [m for m in METRICAS if m in ta.columns and m in tb.columns]

# 1. Médias
comparacao = pd.DataFrame({A: ta[metricas].mean(), B: tb[metricas].mean()})
comparacao.loc["correctness (taxa pass)"] = [ta["correctness_score"].eq("pass").mean(),
                                             tb["correctness_score"].eq("pass").mean()]
comparacao["diferenca"] = comparacao[B] - comparacao[A]
display(comparacao.round(3))

# 2. Taxa de pass por categoria
pass_por_categoria = pd.DataFrame({
    A: ta["correctness_score"].eq("pass").groupby(ta["categoria"]).mean(),
    B: tb["correctness_score"].eq("pass").groupby(tb["categoria"]).mean(),
})
display(pass_por_categoria.round(3))

# 3. Métricas contínuas: vitórias / empates / derrotas de B e Wilcoxon
emparelhado = {}
for m in ["JudgeScore", "AnswerRelevancy", "Faithfulness", "ArgsAdequados", "LatenciaMedia"]:
    pares = pd.concat([ta[m], tb[m]], axis=1, keys=[A, B]).dropna()   # só perguntas com valor nos dois modos
    diferenca = pares[B] - pares[A]
    vitorias, derrotas = int((diferenca > 0).sum()), int((diferenca < 0).sum())
    p = float(wilcoxon(pares[A], pares[B]).pvalue) if vitorias + derrotas > 0 else math.nan   # tudo empatado: não se aplica
    emparelhado[m] = {"n": len(pares), f"vitorias_{B}": vitorias, "empates": int((diferenca == 0).sum()),
                      f"derrotas_{B}": derrotas, "wilcoxon_p": p}

# 4. Correctness: McNemar (só contam as perguntas em que os modos discordam)
validas = ta["correctness_score"].notna() & tb["correctness_score"].notna()
passa_a = ta.loc[validas, "correctness_score"].eq("pass")
passa_b = tb.loc[validas, "correctness_score"].eq("pass")
so_a, so_b = int((passa_a & ~passa_b).sum()), int((~passa_a & passa_b).sum())
p = float(binomtest(so_b, so_a + so_b, 0.5).pvalue) if so_a + so_b > 0 else math.nan
emparelhado["correctness"] = {"n": int(validas.sum()), f"vitorias_{B}": so_b,
                              "empates": int(validas.sum()) - so_a - so_b,
                              f"derrotas_{B}": so_a, "mcnemar_p": p}
display(pd.DataFrame(emparelhado).T)

# 5. Guardar
comparacao.to_csv(os.path.join(PASTA_EXECUCAO, "comparacao.csv"))
guardar_json(os.path.join(PASTA_EXECUCAO, "comparacao.json"), {
    "modos": [A, B],
    "medias": comparacao.round(4).to_dict(orient="index"),
    "pass_por_categoria": pass_por_categoria.round(4).to_dict(orient="index"),
    "emparelhado": emparelhado,
})
print(f"Comparação guardada em {PASTA_EXECUCAO}")

,rag_primeiro,livre,diferenca
ToolHit,0.808,0.577,-0.231
ToolRecall,0.808,0.577,-0.231
ToolPrecision,0.857,0.759,-0.098
NumToolCalls,1.269,0.885,-0.385
ArgsValidos,1.000,1.000,0.000
ArgsAdequados,1.000,1.000,0.000
TaxaSucesso,1.000,0.885,-0.115
LatenciaMedia,0.024,0.335,0.311
JudgeScore,9.269,7.385,-1.885
AnswerRelevancy,0.844,0.789,-0.055


,rag_primeiro,livre
categoria,,
arquivos,1.0,0.438
batcaverna,1.0,1.000
batman,0.0,1.000
web,0.8,1.000


,n,vitorias_livre,empates,derrotas_livre,wilcoxon_p,mcnemar_p
JudgeScore,26.0,1.0,16.0,9.0,0.018671,NaN
AnswerRelevancy,26.0,10.0,4.0,12.0,0.199705,NaN
Faithfulness,15.0,4.0,7.0,4.0,0.887954,NaN
ArgsAdequados,7.0,0.0,7.0,0.0,NaN,NaN
LatenciaMedia,26.0,22.0,0.0,4.0,0.000255,NaN
correctness,26.0,2.0,15.0,9.0,NaN,0.06543


Comparação guardada em /Users/miguelmauricioantonio/Documents/MIA/2/ILN/PROJECT/D2_D3_Miguel_Luís/mcp/avaliacao/resultados/20261004-120944


## 8. Tabelas finais

Resumo para o relatório, organizado por perguntas da avaliação, com os dois modos lado a lado:

1. **Ferramentas:** o agente chama as ferramentas certas?
2. **Servidor MCP:** as ferramentas funcionam, são fiáveis e rápidas?
3. **Correção:** a resposta está correta?
4. **Fundamentação:** a resposta baseia-se no que as ferramentas devolveram?

As médias são por pergunta. `TesteCorreto` e `TesteRespondeu` vêm dos testes diretos da secção 3, por isso são iguais nos dois modos.

Mais duas tabelas: os indicadores principais **por categoria** de pergunta e os indicadores **por ferramenta MCP** (testes diretos e uso pelo agente). Ficam em `resultados/<data-hora>/tabela_final`, `tabela_por_categoria` e `tabela_por_ferramenta` (`.csv` e `.md`).

In [14]:
def taxa_pass(tabela):
    validas = tabela["correctness_score"].dropna()
    return validas.eq("pass").mean() if len(validas) else math.nan


# (grupo, métrica, o que mede, função que calcula o valor a partir da tabela de um modo)
LINHAS_FINAIS = [
    ("1. Ferramentas", "ToolHit", "% de perguntas em que foi chamada pelo menos uma ferramenta esperada",
     lambda t: t["ToolHit"].mean()),
    ("1. Ferramentas", "ToolRecall", "ferramentas esperadas que foram chamadas",
     lambda t: t["ToolRecall"].mean()),
    ("1. Ferramentas", "ToolPrecision", "ferramentas escolhidas pelo LLM que eram esperadas ou permitidas",
     lambda t: t["ToolPrecision"].mean()),
    ("1. Ferramentas", "NumToolCalls", "chamadas a ferramentas por pergunta (média)",
     lambda t: t["NumToolCalls"].mean()),
    ("2. Argumentos", "ArgsValidos", "chamadas do LLM com argumentos válidos para o esquema da ferramenta",
     lambda t: t["ArgsValidos"].mean()),
    ("2. Argumentos", "ArgsAdequados", "chamadas do LLM com argumentos adequados à pergunta (juiz)",
     lambda t: t["ArgsAdequados"].mean()),
    ("3. Servidor MCP", "TesteCorreto", "testes diretos: chamadas com o resultado certo",
     lambda t: testes_df["correto"].mean()),
    ("3. Servidor MCP", "TesteRespondeu", "testes diretos: chamadas a que o servidor respondeu",
     lambda t: testes_df["respondeu"].mean()),
    ("3. Servidor MCP", "TaxaSucesso", "chamadas MCP do agente sem erro",
     lambda t: t["TaxaSucesso"].mean()),
    ("3. Servidor MCP", "LatenciaMedia", "segundos por chamada MCP do agente",
     lambda t: t["LatenciaMedia"].mean()),
    ("4. Correção", "Correctness", "% de respostas pass (Ragas, face à resposta esperada)", taxa_pass),
    ("4. Correção", "JudgeScore", "LLM-as-Judge 0-10 (prompt dos slides)", lambda t: t["JudgeScore"].mean()),
    ("4. Correção", "AnswerRelevancy", "a resposta responde ao que foi perguntado", lambda t: t["AnswerRelevancy"].mean()),
    ("4. Correção", "BLEU", "n-gramas em comum com a resposta esperada", lambda t: t["BLEU"].mean()),
    ("4. Correção", "ROUGE-L", "maior sequência de palavras em comum", lambda t: t["ROUGE-L"].mean()),
    ("5. Fundamentação", "Faithfulness", "afirmações da resposta suportadas pelas saídas das ferramentas",
     lambda t: t["Faithfulness"].mean()),
]

linhas = []
for grupo, metrica, descricao, calcular in LINHAS_FINAIS:
    try:
        valores = {modo: calcular(tabelas[modo]) for modo in MODOS}
    except KeyError:
        continue   # métrica não calculada (por exemplo BLEU sem sacrebleu)
    linhas.append({"grupo": grupo, "métrica": metrica, "o que mede": descricao, **valores})
tabela_final = pd.DataFrame(linhas).set_index(["grupo", "métrica"])

# Causas das respostas erradas
for causa in D3.CAUSAS:
    contagens = {modo: int(tabelas[modo]["causa_falha"].eq(causa).sum()) for modo in MODOS}
    if any(contagens.values()):
        tabela_final.loc[("6. Falhas", causa), :] = {"o que mede": D3.CAUSAS[causa], **contagens}

# Indicadores principais por categoria
por_categoria = {}
for modo in MODOS:
    t = tabelas[modo]
    por_categoria[modo] = pd.DataFrame({
        "n": t.groupby("categoria").size(),
        "ToolHit": t.groupby("categoria")["ToolHit"].mean(),
        "TaxaSucesso": t.groupby("categoria")["TaxaSucesso"].mean(),
        "Correctness": t.groupby("categoria").apply(taxa_pass),
        "JudgeScore": t.groupby("categoria")["JudgeScore"].mean(),
        "Faithfulness": t.groupby("categoria")["Faithfulness"].mean(),
    })
tabela_por_categoria = pd.concat(por_categoria, axis=1)

# Indicadores por ferramenta: testes diretos + uso pelo agente em cada modo
tabela_por_ferramenta = testes_por_ferramenta[["TesteCorreto", "TesteRespondeu", "latencia_media_s"]]
tabela_por_ferramenta = tabela_por_ferramenta.rename(columns={"latencia_media_s": "teste_latencia_s"})
for modo in MODOS:
    uso = chamadas_df[chamadas_df["modo"] == modo].groupby("ferramenta").agg(
        chamadas=("ok", "size"),
        TaxaSucesso=("ok", "mean"),
        LatenciaMedia=("segundos", "mean"),
        ArgsValidos=("ArgsValidos", "mean"),
        ArgsAdequados=("ArgsAdequados", "mean"),
    )
    tabela_por_ferramenta = tabela_por_ferramenta.join(uso.add_prefix(f"{modo}/"), how="outer")


def para_markdown(df):
    """Tabela Markdown simples (sem depender do tabulate)."""
    df = df.reset_index()
    cabecalho = ["/".join(map(str, c)).strip("/") if isinstance(c, tuple) else str(c) for c in df.columns]
    fmt = lambda v: "" if pd.isna(v) else (f"{v:.3f}" if isinstance(v, float) else str(v))
    linhas_md = ["| " + " | ".join(cabecalho) + " |", "|" + "---|" * len(cabecalho)]
    linhas_md += ["| " + " | ".join(fmt(v) for v in linha) + " |" for linha in df.itertuples(index=False)]
    return "\n".join(linhas_md)


tabelas_finais = {"tabela_final": tabela_final, "tabela_por_categoria": tabela_por_categoria,
                  "tabela_por_ferramenta": tabela_por_ferramenta}
for nome, df in tabelas_finais.items():
    df.to_csv(os.path.join(PASTA_EXECUCAO, f"{nome}.csv"))
    with open(os.path.join(PASTA_EXECUCAO, f"{nome}.md"), "w", encoding="utf-8") as f:
        f.write(para_markdown(df) + "\n")

display(tabela_final.style.format({modo: "{:.3f}" for modo in MODOS}, na_rep="—"))
display(tabela_por_categoria.style.format("{:.3f}", na_rep="—").format({(m, "n"): "{:.0f}" for m in MODOS}))
display(tabela_por_ferramenta.style.format("{:.3f}", na_rep="—"))
print(f"Tabelas guardadas em {PASTA_EXECUCAO}")

,TesteCorreto,TesteRespondeu,teste_latencia_s,rag_primeiro/chamadas,rag_primeiro/TaxaSucesso,rag_primeiro/LatenciaMedia,rag_primeiro/ArgsValidos,rag_primeiro/ArgsAdequados,livre/chamadas,livre/TaxaSucesso,livre/LatenciaMedia,livre/ArgsValidos,livre/ArgsAdequados
ferramenta,,,,,,,,,,,,,
clima_gotham,1.000,1.000,0.001,1.000,1.000,0.005,1.000,1.000,1.000,1.000,0.004,1.000,1.000
estado_batmovel,1.000,1.000,0.001,1.000,1.000,0.004,1.000,1.000,1.000,1.000,0.004,1.000,1.000
gerir_agenda_bruce,1.000,1.000,0.001,2.000,1.000,0.004,1.000,1.000,2.000,1.000,0.004,1.000,1.000
ler_pagina,1.000,1.000,0.221,1.000,1.000,0.142,1.000,1.000,3.000,0.333,0.388,1.000,1.000
listar_arquivos,1.000,1.000,0.010,1.000,1.000,0.005,1.000,1.000,1.000,1.000,0.004,1.000,1.000
perguntar_ao_batman,1.000,1.000,0.006,1.000,1.000,0.015,1.000,1.000,1.000,1.000,0.015,1.000,1.000
pesquisar_arquivos,1.000,1.000,0.027,26.000,1.000,0.012,—,—,6.000,0.000,0.004,1.000,1.000
pesquisar_noticias,1.000,1.000,0.658,—,—,—,—,—,1.000,1.000,0.397,1.000,1.000
pesquisar_web,1.000,1.000,2.484,—,—,—,—,—,7.000,1.000,2.895,1.000,1.000


Tabelas guardadas em /Users/miguelmauricioantonio/Documents/MIA/2/ILN/PROJECT/D2_D3_Miguel_Luís/mcp/avaliacao/resultados/20261004-120944
